In [3]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]="platform"
os.environ["JAX_ENABLE_X64"]="false"

import multiprocessing
os.environ["XLA_FLAGS"]="--xla_force_host_platform_device_count={}".format(multiprocessing.cpu_count())
# os.environ["XLA_FLAGS"]="--xla_cpu_multi_thread_eigen=true intra_op_parallelism_threads=16"
import jax
# jax.config.update("jax_log_compiles", True)
# print(multiprocessing.cpu_count())
# print(jax.device_count())
jax.config.update("jax_enable_x64", False)


from sampling import *
from wavefunction import *
from observables import *
from training import *
from utils import *
from fit import *
from fitting_widget import *
import time


from plotly import graph_objects as go
from scipy.optimize import curve_fit



In [ ]:
N = 80
eta = 1
gsq = .73
g = np.sqrt(gsq) # beta = 1/g^2
mu = 0
N_CORES = 16

x = np.random.normal(size=(N, 3))
model = MLP_SO3(hidden_sizes=(N,))

# strip the leading 0 from the string representation of gsq
gsq_str = str(gsq).lstrip("0")
params = np.load(f"data/L_40_g2_{gsq_str}_params.npy", allow_pickle=True).item()
y = model.apply(params,x)

model_apply = jit(model.apply)

psi = model_apply

def gen_init_positions(nchains, input):
    pos_initials = [jax.random.normal(jax.random.PRNGKey(i + int(input)), shape=(N, 3)) for i in range(nchains)]
    # for each chain, for each site, normalize the 3D vector to lie on the unit sphere
    for i, x in enumerate(pos_initials):
        pos_initials[i] /= jnp.linalg.norm(x, axis=-1, keepdims=True)
    return pos_initials


sampler = newSampler(psi, (N, 3))

nchains = N_CORES
pos_initials = gen_init_positions(nchains, int(time.time()))
seeds = jnp.arange(nchains)
var = 0.35

samples, acc_rate = sampler.run_many_chains(params, nchains**4//nchains, 1000, 3, var, pos_initials, seeds)
print(f"Acceptance rate: {acc_rate}")


Acceptance rate: 0.5041720271110535


In [ ]:
samples, acc_rate = sampler.run_many_chains(params, 1000000//16, 50000, 10, var, gen_init_positions(16, int(time.time())), [int(time.time()) + i for i in range(16)])
print(f"Acceptance rate: {acc_rate}")


C, cov, C_uncerts = Cr_with_cov_optimized(samples, batch_size=64)


ui = fitting_widget(
    C,
    cov,
    initial_fit_type="exponential",
    initial_rel_cut=1e-5,
)

In [ ]:
# After clicking "Extract plateau ξ" in the UI:
extracted_xi = ui["result_state"]["xi"]

# print(extracted_xi.mean, extracted_xi.sdev)
print(f"Extracted ξ (g^2 = {gsq}): {extracted_xi}")

In [ ]:
gse, gseu = batched_energy(model, eta, g, mu, params, samples, batch_size=16)
print(f"GSE: {gse} ± {gseu}")